# SQL JOIN 多表连接（练习）

## 0. 环境

In [17]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [2]:
%%sql
CREATE OR REPLACE VIEW sales AS SELECT * FROM '../data/sales.csv';
SELECT COUNT(*) FROM sales;

Running query in 'duckdb:///:memory:'

count_star()
500


## 1. 造练习表
JOIN 需要多张表。我们造一张「客户信息表」customers,和一张「国家信息表」countries:

In [3]:
%%sql
-- 客户信息表:故意让它和 sales 里的 customer_id "不完全对齐"
CREATE OR REPLACE TABLE customers AS
SELECT * FROM (VALUES
    ('C001', 'Alice',   'VIP'),
    ('C002', 'Bob',     'Normal'),
    ('C003', 'Charlie', 'VIP'),
    ('C004', 'Diana',   'Normal'),
    ('C999', 'Eve',     'VIP')      -- C999 在 sales 里不存在(故意的)
) AS t(customer_id, customer_name, level);

SELECT * FROM customers;

Running query in 'duckdb:///:memory:'

customer_id,customer_name,level
C001,Alice,VIP
C002,Bob,Normal
C003,Charlie,VIP
C004,Diana,Normal
C999,Eve,VIP


In [4]:
%%sql
CREATE OR REPLACE TABLE countries AS
SELECT * FROM (VALUES
    ('US',      'North America'),
    ('Germany', 'Europe'),
    ('France',  'Europe'),
    ('Japan',   'Asia')
) AS t(country, region);

SELECT * FROM countries;

Running query in 'duckdb:///:memory:'

country,region
US,North America
Germany,Europe
France,Europe
Japan,Asia


故意制造「不对齐」:customers 里有 C999(sales 里没有),sales 里也有很多 customer_id 不在 customers 这 5 个里。这种不对齐正是 INNER 和 LEFT JOIN 产生差异的根源——下面就靠它看出区别。

## 2. INNER JOIN - 只保留两边都有的

In [5]:
%%sql
-- INNER JOIN：只返回“两张表里都能匹配上”的行
SELECT
    s.order_id,
    s.customer_id,
    c.customer_name,
    c.level,
    s.total
FROM sales s
INNER JOIN customers c
    ON s.customer_id = c.customer_id -- ← 连接条件：靠哪一列对应
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_id,customer_name,level,total
O1001,C004,Diana,Normal,99
O1004,C003,Charlie,VIP,495
O1009,C002,Bob,Normal,297
O1010,C003,Charlie,VIP,1797
O1012,C003,Charlie,VIP,897
O1013,C003,Charlie,VIP,1299
O1016,C004,Diana,Normal,2396
O1019,C003,Charlie,VIP,897
O1022,C002,Bob,Normal,2598
O1024,C004,Diana,Normal,1198


拆解:

FROM sales s —— 主表,s 是别名(省打字)

INNER JOIN customers c —— 要连接的表,别名 c

ON s.customer_id = c.customer_id —— 连接条件:两张表靠哪一列对上

INNER JOIN 的本质:sales 里 customer_id 不在 customers 的行 → 丢掉;customers 里 C999 在 sales 没订单 → 也丢掉。只剩交集。

In [8]:
%%sql
SELECT
    (SELECT COUNT(*) FROM sales) AS sales原始行数,
    (SELECT COUNT(*) FROM sales s INNER JOIN customers c
        ON s.customer_id = c.customer_id) AS inner_join后行数;

Running query in 'duckdb:///:memory:'

sales原始行数,inner_join后行数
500,257


后者明显更少——差额就是「customer_id 匹配不上 customers 的订单」。

## 3. LEFT JOIN - 保留左表全部

In [9]:
%%sql
-- LEFT JOIN：左表（sales）的行全部保留；右表匹配不上的地方填NULL
SELECT 
    s.order_id,
    s.customer_id,
    c.customer_name, -- 匹配不上时这里是NULL
    c.level,
    s.total
FROM sales s
LEFT JOIN customers c
    ON s.customer_id = c.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_id,customer_name,level,total
O1001,C004,Diana,Normal,99
O1004,C003,Charlie,VIP,495
O1009,C002,Bob,Normal,297
O1010,C003,Charlie,VIP,1797
O1012,C003,Charlie,VIP,897
O1013,C003,Charlie,VIP,1299
O1016,C004,Diana,Normal,2396
O1019,C003,Charlie,VIP,897
O1022,C002,Bob,Normal,2598
O1024,C004,Diana,Normal,1198


LEFT JOIN 的本质:左表(FROM 后面那张)一行不丢。右表能匹配就填上,匹配不上就填 NULL。

LEFT JOIN 最经典的用法——找「左表有、右表没有」的孤儿行:

In [10]:
%%sql
-- 找出“有订单、但不在customers表里”的customer_id
SELECT DISTINCT s.customer_id
FROM sales s
LEFT JOIN customers c
    ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL; -- ← 右表列为NULL = 没匹配上

Running query in 'duckdb:///:memory:'

customer_id
C006
C005
C008
C007


💡 LEFT JOIN + WHERE 右表 IS NULL = 反向筛选。这和 Day 7 的 NOT EXISTS 是同一个需求的两种写法——「找左表有右表没有的」。记进翻译表。

⚠️ 注意是 IS NULL 不是 = NULL(Day 4 弱点:NULL 比较必须用 IS)。

## 4. RIGHT JOIN/FULL JOIN

In [12]:
%%sql
-- RIGHT JOIN：右表全部保留（和LEFT镜像）
-- 找出“在customers表里、但从来没下过订单”的客户（比如 C999）
SELECT c.customer_id, c.customer_name
FROM sales s
RIGHT JOIN customers c
    ON s.customer_id = c.customer_id
WHERE s.customer_id IS NULL;

Running query in 'duckdb:///:memory:'

customer_id,customer_name
C999,Eve


In [13]:
%%sql
-- FULL JOIN：两边的行都保留，任意一边匹配不上就填NULL
SELECT s.order_id, s.customer_id, c.customer_name
FROM sales s
FULL JOIN customers c
    ON s.customer_id = c.customer_id
WHERE s.customer_id IS NULL OR c.customer_id IS NULL; -- 两边的“孤儿”都列出来

Running query in 'duckdb:///:memory:'

order_id,customer_id,customer_name
O1000,C007,None
O1002,C005,None
O1003,C007,None
O1005,C008,None
O1006,C005,None
O1007,C005,None
O1008,C007,None
O1011,C007,None
O1014,C008,None
O1015,C005,None


实务提示:RIGHT JOIN 几乎没人用——任何 RIGHT JOIN 都能把两张表对调写成 LEFT JOIN,而 LEFT 更符合「从主表出发」的阅读习惯。知道它存在即可,自己写一律用 LEFT。

## 5. 四种JOIN总览 + 链式JOIN

| JOIN 类型   | 保留谁                     | 一句话         |
|-------------|----------------------------|----------------|
| INNER JOIN  | 只保留两边都匹配上的       | 交集           |
| LEFT JOIN   | 左表全保留 + 右表匹配项    | 左表为主       |
| RIGHT JOIN  | 右表全保留 + 左表匹配项    | 右表为主(少用) |
| FULL JOIN   | 两边全保留                 | 并集           |

链式 JOIN——一次连多张表:

In [16]:
%%sql
-- sales 同时连customers 和 countries
SELECT 
    s.order_id,
    c.customer_name,
    c.level,
    s.country,
    co.region, -- 来自countries表
    s.total
FROM sales s
LEFT JOIN customers c ON s.customer_id = c.customer_id
LEFT JOIN countries co ON s.country = co.country
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_name,level,country,region,total
O1001,Diana,Normal,US,North America,99
O1004,Charlie,VIP,France,Europe,495
O1009,Bob,Normal,Germany,Europe,297
O1016,Diana,Normal,US,North America,2396
O1022,Bob,Normal,France,Europe,2598
O1024,Diana,Normal,France,Europe,1198
O1029,Diana,Normal,Germany,Europe,598
O1031,Alice,VIP,France,Europe,198
O1032,Diana,Normal,US,North America,4995
O1037,Alice,VIP,Germany,Europe,5196


核心心智模型:JOIN 先想「我要不要保留匹配不上的行」——要 → LEFT/FULL,不要 → INNER。